In [84]:
import subprocess
from subprocess import CalledProcessError
import zipfile
from pathlib import Path
from PIL import Image

import torch
import torchvision
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [85]:
# データセットを保存
save_dir = str(Path().resolve()) # 保存ディレクトリをノートブックと同じディレクトリに設定

dataset_dir = save_dir + "/competition_images/"

if Path(dataset_dir).exists():
    print(f"Dataset directory already exists: {dataset_dir}")
else:
    url = "http://10.0.87.42:8080/dataset"
    zip_path = Path(str(save_dir) + "tmp.zip")
    try:
        subprocess.run(["wget", url, "-O", zip_path, "--no-proxy", "-q"], check=True)
    except CalledProcessError:
        url = "https://www.dropbox.com/scl/fi/elofv5nreb3f901orkdae/competition_images.zip?rlkey=vfq7sr4gnejfp60ppt14hm7uu&st=qziwvv00&dl=0"
        subprocess.run(["wget", url, "-O", zip_path, "--no-proxy", "-q"], check=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(save_dir)
        if zip_path.exists():
            zip_path.unlink()
    print(f"Downloaded to {save_dir} and extracted.")


Downloaded to /home/tat/research/others/competition and extracted.


In [86]:
# テストデータ読み込みのためのカスタムデータセットクラス
class TestDataset(Dataset):
    def __init__(self, path, transform=None, target_transform=None):
        self.img_paths = sorted([p for p in Path(path).iterdir()])
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        img_path = self.img_paths[index]
        data = Image.open(img_path).convert('RGB')

        if self.transform:
            data = self.transform(data)
        if self.target_transform:
            path = self.target_transform(path)
        return data, str(img_path.name)

    def __len__(self):
        return len(self.img_paths)


In [87]:
# ハイパーパラメータを指定
epochs = 10
lr = 0.001
batch_size = 128

In [88]:
# モデルと訓練に必要なものを定義
num_classes = 4

train_tf = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.3, 0.3, 0.3], std=[0.3, 0.3, 0.3])
])
test_tf = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.3, 0.3, 0.3], std=[0.3, 0.3, 0.3])
])

train_ds = ImageFolder(root=dataset_dir + "train_val", transform=train_tf)
test_ds = TestDataset(path=dataset_dir + "test", transform=test_tf)
model = torchvision.models.resnet18(num_classes=num_classes)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)


In [89]:
# 訓練フローを定義
def train_1epoch(model, train_dl, criterion, optimizer, device):
    total_loss = 0.0
    total_corr = 0

    model.train()
    for inputs, labels in train_dl:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(inputs)

        loss = criterion(outputs, labels)
        preds = torch.argmax(outputs.detach(), dim=1)
        corr = torch.sum(preds == labels.data).item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_corr += corr
    train_loss = total_loss / len(train_dl.dataset)
    train_acc = total_corr / len(train_dl.dataset)

    return train_loss, train_acc

In [90]:
# 推論フローを定義
def pred(model, test_dl, device, categorize=True):
    total_outputs = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for inputs, labels in test_dl:
            inputs = inputs.to(device)

            outputs = model(inputs).detach()

            if categorize:
                outputs = torch.argmax(outputs, dim=1)

            total_outputs.append(outputs)
            all_labels.extend(labels)

    outputs = torch.cat(total_outputs, dim=0).cpu().tolist()
    return outputs, all_labels

In [91]:
# 訓練
model.to(device)
for epoch in range(epochs):
    train_loss, train_acc = train_1epoch(model, train_dl, criterion, optimizer, device)
    print(f"epoch {epoch+1:>3}/{epochs:>3}: train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}")


epoch   1/ 10: train_loss: 0.0092, train_acc: 0.6092
epoch   2/ 10: train_loss: 0.0050, train_acc: 0.7708
epoch   3/ 10: train_loss: 0.0041, train_acc: 0.8350
epoch   4/ 10: train_loss: 0.0038, train_acc: 0.8192
epoch   5/ 10: train_loss: 0.0035, train_acc: 0.8342
epoch   6/ 10: train_loss: 0.0029, train_acc: 0.8725
epoch   7/ 10: train_loss: 0.0027, train_acc: 0.8733
epoch   8/ 10: train_loss: 0.0028, train_acc: 0.8675
epoch   9/ 10: train_loss: 0.0025, train_acc: 0.9008
epoch  10/ 10: train_loss: 0.0022, train_acc: 0.9125


In [92]:
# 推論・結果をCSV形式で保存
csv_name = "predictions.csv"

outputs, all_labels = pred(model, test_dl, device)
lines = [f"{l},{o}" for o, l in zip(outputs, all_labels)]
csv_text = "\n".join(lines)

with open(csv_name, "w", encoding="utf-8") as f:
    f.write(csv_text)